# Sport Data Solution — POC Dispositif Avantages Sportifs
## Cahier d'Exploration des Sources, Validation Géospatiale et Recette Analytique

**Périmètre documentaire :** Audit technique des référentiels RH et déclaratifs sportifs, validation des règles d'éligibilité aux avantages sociaux (prime de mobilité active et journées bien-être) et contrôle de cohérence des indicateurs exposés à la couche de restitution décisionnelle.

**Référentiel d'entreprise :** Siège social situé au 1362 Avenue des Platanes, 34970 Lattes.

## 1. Configuration de l'environnement et chargement des paramètres

In [1]:
# Initialisation du contexte d'exécution et lecture de la configuration métier
import io
import json
import unicodedata
from contextlib import closing
from decimal import Decimal

import openpyxl
import pg8000.dbapi

from src.ingestion.minio_loader import build_s3_client
from src.settings import EnvSettings, load_business_rules

settings = EnvSettings()
rules = load_business_rules()

walk_cap = rules.mobility_bonus.eligible_modes["Marche/running"].max_distance_km
bike_cap = rules.mobility_bonus.eligible_modes["Vélo/Trottinette/Autres"].max_distance_km
prime_pct = rules.mobility_bonus.annual_gross_salary_rate * 100

with closing(pg8000.dbapi.connect(**settings.pg8000_connect_kwargs("pipeline"))) as pg_conn:
    pg_cur = pg_conn.cursor()
    pg_cur.execute("SELECT current_database(), current_user, current_setting('wal_level');")
    db_name, db_user, wal_level = pg_cur.fetchone()
    pg_cur.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name IN ('gold_mart', 'hr_restricted', 'oltp_sports', 'ops_monitoring')
        ORDER BY schema_name;
        """
    )
    active_schemas = [row[0] for row in pg_cur.fetchall()]

s3_client = build_s3_client(settings)
s3_buckets = sorted(b["Name"] for b in s3_client.list_buckets().get("Buckets", []))

print(f"Entreprise          : {rules.company.name} ({rules.company.headquarters_address})")
print(
    f"Prime mobilité      : {prime_pct:.1f} % du salaire brut annuel "
    f"(Marche <= {walk_cap:.1f} km | Vélo <= {bike_cap:.1f} km)"
)
print(
    f"Journées bien-être  : {rules.wellbeing_days.granted_days_count} jours accordés dès "
    f"{rules.wellbeing_days.min_activities_per_year} activités / "
    f"{rules.wellbeing_days.rolling_window_months} mois glissants"
)
print(
    f"PostgreSQL 16       : db={db_name} | role={db_user} | wal_level={wal_level} | "
    f"schemas={active_schemas}"
)
print(f"Buckets S3 MinIO    : {s3_buckets}")

Entreprise          : Sport Data Solution (1362 Av. des Platanes, 34970 Lattes, France)
Prime mobilité      : 5.0 % du salaire brut annuel (Marche <= 15.0 km | Vélo <= 25.0 km)
Journées bien-être  : 5 jours accordés dès 15 activités / 12 mois glissants
PostgreSQL 16       : db=sds_platform | role=sds_pipeline | wal_level=logical | schemas=['gold_mart', 'hr_restricted', 'oltp_sports', 'ops_monitoring']
Buckets S3 MinIO    : ['sds-bronze', 'sds-gold', 'sds-raw-referentials', 'sds-silver']


## 2. Profilage et audit d'intégrité des référentiels sources (RH et Pratique Sportive)

In [2]:
# Chargement et contrôle d'unicité des identifiants collaborateurs
def summarize_s3_workbook(s3_key: str) -> tuple[int, set[int], str]:
    obj = s3_client.get_object(Bucket=settings.minio_bucket_raw, Key=s3_key)
    payload = obj["Body"].read()
    sha_prefix = obj.get("Metadata", {}).get("sha256", "n/a")[:12]
    wb = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()
    headers = [unicodedata.normalize("NFC", str(col).strip()) for col in rows[0]]
    id_idx = headers.index("ID salarié")
    emp_ids = {
        int(round(float(row[id_idx])))
        for row in rows[1:]
        if row[id_idx] is not None and str(row[id_idx]).strip() != ""
    }
    non_empty_rows = sum(1 for row in rows[1:] if any(cell is not None for cell in row))
    return non_empty_rows, emp_ids, sha_prefix


hr_rows_cnt, hr_ids, hr_sha = summarize_s3_workbook(settings.hr_referential_s3_key)
sp_rows_cnt, sp_ids, sp_sha = summarize_s3_workbook(settings.sports_referential_s3_key)

print("=== Intégrité des référentiels bruts S3 (sans exposition PII) ===")
print(
    f"RH      : {hr_rows_cnt} lignes | {len(hr_ids)} IDs uniques | "
    f"sha256={hr_sha}... | doublons={hr_rows_cnt - len(hr_ids)}"
)
print(
    f"Sportif : {sp_rows_cnt} lignes | {len(sp_ids)} IDs uniques | "
    f"sha256={sp_sha}... | écart_IDs_RH_Sport={len(hr_ids ^ sp_ids)}"
)

with closing(pg8000.dbapi.connect(**settings.pg8000_connect_kwargs("pipeline"))) as conn:
    cur = conn.cursor()
    cur.execute(
        """
        SELECT business_unit,
               COUNT(*) AS effectif,
               SUM(CASE WHEN contract_type = 'CDI' THEN 1 ELSE 0 END) AS cdi_count,
               SUM(CASE WHEN contract_type = 'CDD' THEN 1 ELSE 0 END) AS cdd_count,
               MIN(hire_date),
               MAX(hire_date)
        FROM hr_restricted.employees_audit
        GROUP BY business_unit
        ORDER BY effectif DESC, business_unit ASC;
        """
    )
    bu_rows = cur.fetchall()

print("\n=== Répartition agrégée des effectifs par Business Unit ===")
print(f"{'Business Unit':<16} | {'Total':>5} | {'CDI':>4} | {'CDD':>4} | {'Période embauche'}")
print("-" * 62)
for bu, total, cdi, cdd, min_hire, max_hire in bu_rows:
    print(f"{bu:<16} | {total:>5} | {cdi:>4} | {cdd:>4} | {min_hire} -> {max_hire}")

with closing(pg8000.dbapi.connect(**settings.pg8000_connect_kwargs("bi_reader"))) as bi_conn:
    bi_cur = bi_conn.cursor()
    rbac_blocked: dict[str, bool] = {}
    for schema_table in ("hr_restricted.employees_audit", "oltp_sports.sport_activities"):
        try:
            bi_cur.execute(f"SELECT 1 FROM {schema_table} LIMIT 1;")
            rbac_blocked[schema_table] = False
        except pg8000.dbapi.DatabaseError as exc:
            bi_conn.rollback()
            rbac_blocked[schema_table] = "42501" in str(exc)

print(f"\nCloisonnement RBAC sds_bi_reader (SQLSTATE 42501) : {rbac_blocked}")

=== Intégrité des référentiels bruts S3 (sans exposition PII) ===
RH      : 161 lignes | 161 IDs uniques | sha256=58a652ecc510... | doublons=0
Sportif : 161 lignes | 161 IDs uniques | sha256=a40695b63142... | écart_IDs_RH_Sport=0

=== Répartition agrégée des effectifs par Business Unit ===
Business Unit    | Total |  CDI |  CDD | Période embauche
--------------------------------------------------------------
Finance          |    42 |   39 |    3 | 2020-05-19 -> 2025-02-27
Support          |    35 |   33 |    2 | 2020-03-31 -> 2025-03-24
Ventes           |    33 |   30 |    3 | 2020-05-26 -> 2025-03-24
R&D              |    26 |   26 |    0 | 2020-07-07 -> 2024-10-25
Marketing        |    25 |   21 |    4 | 2020-04-21 -> 2025-01-24

Cloisonnement RBAC sds_bi_reader (SQLSTATE 42501) : {'hr_restricted.employees_audit': True, 'oltp_sports.sport_activities': True}


## 3. Contrôle de cohérence géospatiale des déclarations domicile-travail

In [3]:
# Analyse des distances calculées via Google Maps et isolement des anomalies déclaratives
cache_path = settings.commute_cache_path()
cache_raw = cache_path.read_text(encoding="utf-8")
cache_entries = json.loads(cache_raw).get("entries", {})
pii_leak = any(city in cache_raw.lower() for city in ("lattes", "montpellier", "pérols"))

print(
    f"Cache géospatial RGPD : {len(cache_entries)} trajets hachés (SHA-256 salé) | "
    f"fuite_adresse_clair={pii_leak}"
)

prime_rate = Decimal(str(rules.mobility_bonus.annual_gross_salary_rate))

with closing(pg8000.dbapi.connect(**settings.pg8000_connect_kwargs("pipeline"))) as conn:
    cur = conn.cursor()
    cur.execute(
        """
        SELECT declared_commute_mode,
               COUNT(*) AS effectif,
               SUM(CASE WHEN is_mobility_bonus_eligible THEN 1 ELSE 0 END) AS eligibles,
               SUM(CASE WHEN is_commute_anomaly THEN 1 ELSE 0 END) AS anomalies,
               ROUND(AVG(computed_distance_km), 2) AS avg_km,
               MAX(computed_distance_km) AS max_km,
               ROUND(
                   SUM(
                       CASE
                           WHEN is_mobility_bonus_eligible
                           THEN annual_gross_salary_eur * %s
                           ELSE 0
                       END
                   ),
                   2
               ) AS budget_prime_eur
        FROM hr_restricted.employees_audit
        GROUP BY declared_commute_mode
        ORDER BY effectif DESC;
        """,
        (prime_rate,),
    )
    commute_summary = cur.fetchall()

    cur.execute(
        """
        SELECT declared_commute_mode,
               computed_distance_km,
               max_allowed_distance_km,
               anomaly_reason
        FROM hr_restricted.employees_audit
        WHERE is_commute_anomaly = TRUE;
        """
    )
    anomalies_rows = cur.fetchall()

print("\n=== Synthèse d'éligibilité à la prime de mobilité par mode déclaré ===")
header_c6 = (
    f"{'Mode déclaré':<30} | {'Total':>5} | {'Élig.':>5} | {'Anom.':>5} | "
    f"{'Moy km':>6} | {'Max km':>6} | {'Budget EUR':>11}"
)
print(header_c6)
print("-" * len(header_c6))
total_budget = Decimal("0.00")
for mode, tot, elig, anom, avg_km, max_km, budget_eur in commute_summary:
    total_budget += budget_eur or Decimal("0.00")
    avg_s = f"{avg_km:.2f}" if avg_km is not None else "  N/A "
    max_s = f"{max_km:.3f}" if max_km is not None else "  N/A "
    print(
        f"{mode:<30} | {tot:>5} | {elig:>5} | {anom:>5} | "
        f"{avg_s:>6} | {max_s:>6} | {budget_eur:>11,.2f}"
    )
print(f"\nEnveloppe annuelle prime de mobilité validée (taux 5,0 %) : {total_budget:,.2f} EUR")

print(f"\n=== Anomalies déclaratives isolées ({len(anomalies_rows)}, vue anonymisée) ===")
for mode, dist_km, cap_km, reason in anomalies_rows:
    print(f"- Mode={mode} | distance={dist_km} | plafond={cap_km} km | motif={reason}")

Cache géospatial RGPD : 68 trajets hachés (SHA-256 salé) | fuite_adresse_clair=False

=== Synthèse d'éligibilité à la prime de mobilité par mode déclaré ===
Mode déclaré                   | Total | Élig. | Anom. | Moy km | Max km |  Budget EUR
--------------------------------------------------------------------------------------
véhicule thermique/électrique  |    73 |     0 |     0 |   N/A  |   N/A  |        0.00
Vélo/Trottinette/Autres        |    54 |    53 |     1 |   8.61 | 18.579 |  131,015.00
Transports en commun           |    20 |     0 |     0 |   N/A  |   N/A  |        0.00
Marche/running                 |    14 |    14 |     0 |   2.43 |  4.242 |   37,809.50

Enveloppe annuelle prime de mobilité validée (taux 5,0 %) : 168,824.50 EUR

=== Anomalies déclaratives isolées (1, vue anonymisée) ===
- Mode=Vélo/Trottinette/Autres | distance=None | plafond=25.00 km | motif=Distance domicile-travail indisponible ou invalide


## 4. Analyse statistique des flux d'activités physiques sur 12 mois glissants

In [4]:
# Distribution des volumes d'activités par discipline, unité opérationnelle
# et éligibilité aux journées bien-être
min_acts = rules.wellbeing_days.min_activities_per_year
granted_days = rules.wellbeing_days.granted_days_count

with closing(pg8000.dbapi.connect(**settings.pg8000_connect_kwargs("pipeline"))) as conn:
    cur = conn.cursor()
    cur.execute(
        """
        SELECT COUNT(*),
               COUNT(DISTINCT sa.employee_id),
               COUNT(DISTINCT sa.sport_type),
               SUM(CASE WHEN sa.sport_type = 'Runing' THEN 1 ELSE 0 END),
               SUM(CASE WHEN sa.activity_start::date < ea.hire_date THEN 1 ELSE 0 END),
               MIN(sa.activity_start)::date,
               MAX(sa.activity_end)::date
        FROM oltp_sports.sport_activities sa
        JOIN hr_restricted.employees_audit ea ON ea.employee_id = sa.employee_id;
        """
    )
    tot_acts, active_emps, nb_sports, typo_cnt, pre_hire_cnt, d_min, d_max = cur.fetchone()

    cur.execute(
        """
        SELECT sport_type,
               COUNT(*) AS seances,
               COUNT(DISTINCT employee_id) AS pratiquants,
               ROUND(AVG(elapsed_seconds) / 60.0, 1) AS duree_moy_min,
               ROUND(AVG(distance_m) / 1000.0, 2) AS dist_moy_km
        FROM oltp_sports.sport_activities
        GROUP BY sport_type
        ORDER BY seances DESC, sport_type ASC;
        """
    )
    sport_rows = cur.fetchall()

    cur.execute(
        """
        WITH emp_volumes AS (
            SELECT ea.employee_id,
                   ea.business_unit,
                   COUNT(sa.activity_id) AS nb_acts
            FROM hr_restricted.employees_audit ea
            LEFT JOIN oltp_sports.sport_activities sa
                   ON sa.employee_id = ea.employee_id
                  AND sa.activity_start >= NOW() - INTERVAL '12 months'
            GROUP BY ea.employee_id, ea.business_unit
        )
        SELECT business_unit,
               COUNT(*) AS effectif_total,
               SUM(CASE WHEN nb_acts > 0 THEN 1 ELSE 0 END) AS pratiquants,
               SUM(nb_acts) AS total_seances,
               SUM(CASE WHEN nb_acts >= %s THEN 1 ELSE 0 END) AS eligibles_bien_etre
        FROM emp_volumes
        GROUP BY business_unit
        ORDER BY eligibles_bien_etre DESC, business_unit ASC;
        """,
        (min_acts,),
    )
    bu_sport_rows = cur.fetchall()

total_wellbeing_emps = sum(int(row[4]) for row in bu_sport_rows)
print(
    f"Flux 12 mois [{d_min} -> {d_max}] : {tot_acts} séances | {active_emps} pratiquants | "
    f"{nb_sports} disciplines"
)
print(
    f"Contrôles qualité OLTP : reliquat_Runing={typo_cnt} | avant_embauche={pre_hire_cnt} | "
    f"éligibles_bien_être={total_wellbeing_emps} ({total_wellbeing_emps * granted_days} jours)"
)

print("\n=== Volume et métriques moyennes par discipline normalisée ===")
hdr_sp = (
    f"{'Discipline':<18} | {'Séances':>7} | {'Pratiq.':>7} | "
    f"{'Durée min':>9} | {'Dist. km':>8}"
)
print(hdr_sp)
print("-" * len(hdr_sp))
for sport, cnt, emps, avg_min, avg_km in sport_rows:
    dist_str = f"{avg_km:.2f}" if avg_km is not None else "N/A"
    print(f"{sport:<18} | {cnt:>7} | {emps:>7} | {avg_min:>9.1f} | {dist_str:>8}")

print(f"\n=== Éligibilité aux journées bien-être (>={min_acts} séances/an) par BU ===")
hdr_bu = (
    f"{'Business Unit':<16} | {'Effectif':>8} | {'Sportifs':>8} | "
    f"{'Séances':>7} | {'Éligibles':>9}"
)
print(hdr_bu)
print("-" * len(hdr_bu))
for bu, eff, prat, acts, elig in bu_sport_rows:
    print(f"{bu:<16} | {eff:>8} | {prat:>8} | {acts:>7} | {elig:>9}")

Flux 12 mois [2025-12-05 -> 2026-11-17] : 1723 séances | 95 pratiquants | 15 disciplines
Contrôles qualité OLTP : reliquat_Runing=0 | avant_embauche=0 | éligibles_bien_être=49 (245 jours)

=== Volume et métriques moyennes par discipline normalisée ===
Discipline         | Séances | Pratiq. | Durée min | Dist. km
-------------------------------------------------------------
Randonnée          |     361 |      16 |     164.9 |    12.30
Course à pied      |     340 |      18 |      52.1 |     9.67
Tennis             |     176 |      11 |      82.0 |      N/A
Natation           |     162 |       8 |      48.7 |     2.72
Football           |     127 |       6 |      82.6 |      N/A
Rugby              |     103 |       6 |      86.7 |      N/A
Boxe               |      94 |       4 |      66.7 |      N/A
Badminton          |      78 |       5 |      69.9 |      N/A
Judo               |      50 |       4 |      69.6 |      N/A
Voile              |      50 |       5 |     157.4 |    30.75
Bask

## 5. Recette des contrôles de qualité des données et supervision des exécutions

In [5]:
# Synthèse des résultats de validation automatisée et métriques de volumétrie du pipeline
from deltalake import DeltaTable

from src.processing.lakehouse_bronze_silver import read_delta_table
from src.settings import LakehouseLayer

storage_opts = settings.delta_storage_options()

with closing(pg8000.dbapi.connect(**settings.pg8000_connect_kwargs("bi_reader"))) as bi_conn:
    bi_cur = bi_conn.cursor()

    bi_cur.execute(
        """
        SELECT run_id::text, rows_processed, duration_ms, finished_at::date
        FROM ops_monitoring.pipeline_runs
        WHERE pipeline_name = 'soda_data_quality_gate'
          AND status = 'SUCCESS'
        ORDER BY finished_at DESC
        LIMIT 1;
        """
    )
    latest_q_run_id, q_rows_eval, q_dur_ms, q_date = bi_cur.fetchone()

    bi_cur.execute(
        """
        SELECT layer_name,
               COUNT(*) AS total_checks,
               SUM(CASE WHEN outcome = 'PASS' THEN 1 ELSE 0 END) AS pass_cnt,
               SUM(CASE WHEN outcome = 'WARN' THEN 1 ELSE 0 END) AS warn_cnt,
               SUM(CASE WHEN outcome = 'FAIL' THEN 1 ELSE 0 END) AS fail_cnt,
               SUM(failed_rows_count) AS failed_rows
        FROM ops_monitoring.quality_check_results
        WHERE run_id = CAST(%s AS UUID)
        GROUP BY layer_name
        ORDER BY CASE layer_name
            WHEN 'hr_audit' THEN 1
            WHEN 'bronze' THEN 2
            WHEN 'silver' THEN 3
            WHEN 'gold' THEN 4
            ELSE 5
        END;
        """,
        (latest_q_run_id,),
    )
    quality_layer_rows = bi_cur.fetchall()

    bi_cur.execute(
        """
        WITH ranked_runs AS (
            SELECT pipeline_name,
                   status,
                   rows_processed,
                   anomalies_detected,
                   duration_ms,
                   ROW_NUMBER() OVER (
                       PARTITION BY pipeline_name
                       ORDER BY started_at DESC
                   ) AS rn
            FROM ops_monitoring.pipeline_runs
        ),
        agg_runs AS (
            SELECT pipeline_name,
                   COUNT(*) AS total_runs,
                   SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS ok_runs
            FROM ops_monitoring.pipeline_runs
            GROUP BY pipeline_name
        )
        SELECT a.pipeline_name,
               a.total_runs,
               a.ok_runs,
               r.status AS last_status,
               r.rows_processed AS last_rows,
               r.anomalies_detected AS last_anomalies,
               r.duration_ms AS last_duration_ms
        FROM agg_runs a
        JOIN ranked_runs r
          ON r.pipeline_name = a.pipeline_name
         AND r.rn = 1
        ORDER BY a.pipeline_name ASC;
        """
    )
    pipeline_obs_rows = bi_cur.fetchall()

    bi_cur.execute(
        """
        SELECT total_employees,
               total_valid_activities_12m,
               total_mobility_bonus_eur,
               total_wellbeing_days_granted
        FROM gold_mart.kpi_executive_snapshot;
        """
    )
    mart_emp, mart_act, mart_bonus, mart_wb_days = bi_cur.fetchone()

medallion_specs: list[tuple[LakehouseLayer, str]] = [
    ("bronze", settings.bronze_employees_table),
    ("bronze", settings.bronze_activities_table),
    ("silver", settings.silver_employees_table),
    ("silver", settings.silver_activities_table),
    ("gold", settings.gold_employee_benefits_table),
    ("gold", settings.gold_bu_summary_table),
    ("gold", settings.gold_sports_monthly_table),
    ("gold", settings.gold_executive_kpis_table),
]

forbidden_pii_columns = {
    "employee_id",
    "first_name",
    "last_name",
    "employee_full_name",
    "birth_date",
    "home_address",
    "comment",
}

delta_inventory: list[tuple[str, str, int, int, int]] = []
detected_pii_cols: set[str] = set()
gold_s3_snapshot: dict[str, object] = {}

for layer_name, table_name in medallion_specs:
    uri = settings.lakehouse_table_uri(layer_name, table_name)
    dt_version = int(DeltaTable(uri, storage_options=storage_opts).version())
    df_tbl = read_delta_table(uri, storage_opts)
    detected_pii_cols.update(forbidden_pii_columns.intersection(df_tbl.columns))
    delta_inventory.append(
        (layer_name, table_name, dt_version, df_tbl.height, len(df_tbl.columns))
    )
    if table_name == settings.gold_executive_kpis_table:
        gold_s3_snapshot = df_tbl.to_dicts()[0]

tot_checks = sum(int(r[1]) for r in quality_layer_rows)
tot_pass = sum(int(r[2]) for r in quality_layer_rows)
tot_fail = sum(int(r[4]) for r in quality_layer_rows)

s3_emp = int(gold_s3_snapshot["total_employees"])
s3_act = int(gold_s3_snapshot["total_valid_activities_12m"])
s3_bonus = float(gold_s3_snapshot["total_mobility_bonus_eur"])
s3_wb = int(gold_s3_snapshot["total_wellbeing_days_granted"])
pg_bonus = float(mart_bonus)

print(
    f"Portail qualité Soda Core ({q_date}) : {tot_pass}/{tot_checks} contrôles PASS "
    f"(échecs={tot_fail}) | {q_rows_eval} lignes évaluées en {q_dur_ms} ms"
)
print(
    f"Audit Zéro-PII Lakehouse (8 tables Delta S3) : "
    f"{len(detected_pii_cols)} colonne nominative exposée"
)
print(
    f"Réconciliation S3 Gold vs gold_mart : effectif={s3_emp}/{mart_emp} | "
    f"séances={s3_act}/{mart_act} | prime={s3_bonus:.2f}/{pg_bonus:.2f} EUR | "
    f"jours={s3_wb}/{mart_wb_days}"
)

print("\n=== Résultats des contrats SodaCL par couche (rôle lecture sds_bi_reader) ===")
hdr_q = (
    f"{'Couche':<12} | {'Contrats':>8} | {'PASS':>6} | "
    f"{'WARN':>6} | {'FAIL':>6} | {'Lignes rejetées':>15}"
)
print(hdr_q)
print("-" * len(hdr_q))
for layer, cnt, p_cnt, w_cnt, f_cnt, f_rows in quality_layer_rows:
    print(
        f"{layer:<12} | {int(cnt):>8} | {int(p_cnt):>6} | "
        f"{int(w_cnt):>6} | {int(f_cnt):>6} | {int(f_rows):>15}"
    )

print("\n=== Inventaire et versionnage des tables Delta Lake sur MinIO S3 ===")
hdr_dl = f"{'Couche':<8} | {'Table Delta':<26} | {'Version':>7} | {'Lignes':>7} | {'Colonnes':>8}"
print(hdr_dl)
print("-" * len(hdr_dl))
for layer_name, table_name, ver, rows_cnt, cols_cnt in delta_inventory:
    print(f"{layer_name:<8} | {table_name:<26} | {ver:>7} | {rows_cnt:>7} | {cols_cnt:>8}")

print("\n=== Supervision des exécutions enregistrées dans ops_monitoring.pipeline_runs ===")
hdr_ops = (
    f"{'Pipeline':<28} | {'Runs (OK/Tot)':>13} | {'Dernier état':<12} | "
    f"{'Lignes':>7} | {'Anom.':>5} | {'Durée ms':>8}"
)
print(hdr_ops)
print("-" * len(hdr_ops))
for p_name, tot_r, ok_r, last_st, last_r, last_anom, last_ms in pipeline_obs_rows:
    ratio_txt = f"{int(ok_r)}/{int(tot_r)}"
    ms_val = int(last_ms) if last_ms is not None else 0
    print(
        f"{p_name:<28} | {ratio_txt:>13} | {last_st:<12} | "
        f"{int(last_r):>7} | {int(last_anom):>5} | {ms_val:>8}"
    )


Portail qualité Soda Core (2026-11-20) : 22/22 contrôles PASS (échecs=0) | 4096 lignes évaluées en 4213 ms
Audit Zéro-PII Lakehouse (8 tables Delta S3) : 0 colonne nominative exposée
Réconciliation S3 Gold vs gold_mart : effectif=161/161 | séances=1723/1723 | prime=168824.50/168824.50 EUR | jours=245/245

=== Résultats des contrats SodaCL par couche (rôle lecture sds_bi_reader) ===
Couche       | Contrats |   PASS |   WARN |   FAIL | Lignes rejetées
--------------------------------------------------------------------
hr_audit     |        6 |      6 |      0 |      0 |               0
bronze       |        4 |      4 |      0 |      0 |               0
silver       |        7 |      7 |      0 |      0 |               0
gold         |        5 |      5 |      0 |      0 |               0

=== Inventaire et versionnage des tables Delta Lake sur MinIO S3 ===
Couche   | Table Delta                | Version |  Lignes | Colonnes
--------------------------------------------------------------

## 6. Simulation budgétaire paramétrique et réconciliation des KPI décisionnels

In [ ]:
# Calcul d'impact financier selon les hypothèses de taux de prime et seuils d'éligibilité
